# Notebook 2: Analyze Visium Fluorescence Data with Squidpy

**Dataset:** Visium fluorescence slide of mouse brain coronal section (pre-processed crop)

**Reference:** [Squidpy Visium Fluorescence Tutorial](https://squidpy.readthedocs.io/en/stable/notebooks/tutorials/tutorial_visium_fluo.html)

This notebook demonstrates image analysis features of Squidpy for Visium fluorescence data, including image segmentation, segmentation features, and image feature extraction/clustering.

## 1. Setup & Imports

In [ ]:
%matplotlib inline

import pandas as pd
import matplotlib.pyplot as plt
import anndata as ad
import scanpy as sc
import squidpy as sq

sc.logging.print_header()
print(f"squidpy=={sq.__version__}")

## 2. Load Pre-processed Data

In [ ]:
# Load pre-processed dataset and tissue image
img = sq.datasets.visium_fluo_image_crop()
adata = sq.datasets.visium_fluo_adata_crop()

## 3. Visualize Cluster Annotation

In [ ]:
sq.pl.spatial_scatter(adata, color="cluster")
plt.savefig("../results/02_visium_fluo/01_spatial_clusters.png", dpi=150, bbox_inches="tight")
plt.show()

## 4. Visualize Fluorescence Channels

The fluorescence image has three channels:
- **DAPI** (specific to DNA)
- **anti-NEUN** (specific to neurons)
- **anti-GFAP** (specific to Glial cells)

In [ ]:
img.show(channelwise=True)
plt.savefig("../results/02_visium_fluo/02_fluorescence_channels.png", dpi=150, bbox_inches="tight")
plt.show()

## 5. Image Segmentation

Pre-process the image by smoothing, then apply watershed segmentation on the DAPI channel.

In [ ]:
sq.im.process(
    img=img,
    layer="image",
    method="smooth",
)

sq.im.segment(img=img, layer="image_smooth", method="watershed", channel=0, chunks=1000)

In [ ]:
# Visualize segmentation result
fig, ax = plt.subplots(1, 2)
img_crop = img.crop_corner(2000, 2000, size=500)
img_crop.show(layer="image", channel=0, ax=ax[0])
img_crop.show(
    layer="segmented_watershed",
    channel=0,
    ax=ax[1],
)
plt.savefig("../results/02_visium_fluo/03_segmentation.png", dpi=150, bbox_inches="tight")
plt.show()

## 6. Segmentation Features

Calculate segmentation features: cell count per spot, mean intensities per channel.

In [ ]:
features_kwargs = {"segmentation": {"label_layer": "segmented_watershed"}}

sq.im.calculate_image_features(
    adata,
    img,
    features="segmentation",
    layer="image",
    key_added="features_segmentation",
    n_jobs=1,
    features_kwargs=features_kwargs,
)

In [ ]:
sq.pl.spatial_scatter(
    sq.pl.extract(adata, "features_segmentation"),
    color=[
        "segmentation_label",
        "cluster",
        "segmentation_ch-0_mean_intensity_mean",
        "segmentation_ch-1_mean_intensity_mean",
    ],
    frameon=False,
    ncols=2,
)
plt.savefig("../results/02_visium_fluo/04_segmentation_features.png", dpi=150, bbox_inches="tight")
plt.show()

## 7. Extract and Cluster Image Features

Calculate summary, histogram, and texture features at multiple scales.

In [ ]:
params = {
    "features_orig": {
        "features": ["summary", "texture", "histogram"],
        "scale": 1.0,
        "mask_circle": True,
    },
    "features_context": {"features": ["summary", "histogram"], "scale": 1.0},
    "features_lowres": {"features": ["summary", "histogram"], "scale": 0.25},
}

for feature_name, cur_params in params.items():
    sq.im.calculate_image_features(
        adata, img, layer="image", key_added=feature_name, n_jobs=1, **cur_params
    )

adata.obsm["features"] = pd.concat(
    [adata.obsm[f] for f in params.keys()], axis="columns"
)
adata.obsm["features"].columns = ad.utils.make_index_unique(
    adata.obsm["features"].columns
)

In [ ]:
def cluster_features(features: pd.DataFrame, like=None):
    """Calculate leiden clustering of features."""
    if like is not None:
        features = features.filter(like=like)
    adata = ad.AnnData(features)
    sc.pp.scale(adata)
    sc.pp.pca(adata, n_comps=min(10, features.shape[1] - 1))
    sc.pp.neighbors(adata)
    sc.tl.leiden(adata)
    return adata.obs["leiden"]

In [ ]:
adata.obs["features_summary_cluster"] = cluster_features(
    adata.obsm["features"], like="summary"
)
adata.obs["features_histogram_cluster"] = cluster_features(
    adata.obsm["features"], like="histogram"
)
adata.obs["features_texture_cluster"] = cluster_features(
    adata.obsm["features"], like="texture"
)

sc.set_figure_params(facecolor="white", figsize=(8, 8))
sq.pl.spatial_scatter(
    adata,
    color=[
        "features_summary_cluster",
        "features_histogram_cluster",
        "features_texture_cluster",
        "cluster",
    ],
    ncols=3,
)
plt.savefig("../results/02_visium_fluo/05_feature_clusters.png", dpi=150, bbox_inches="tight")
plt.show()

## Summary

- Loaded pre-processed Visium fluorescence mouse brain data (3 channels: DAPI, NEUN, GFAP)
- Performed watershed segmentation on DAPI channel
- Extracted segmentation features (cell count, channel intensities per spot)
- Computed summary, histogram, and texture features at multiple scales
- Clustered image features and compared with gene-space clusters
- Image feature clusters reveal finer spatial structure (e.g., Hippocampus sub-regions)